# Pre-trained MolFormer embeddings demo

This notebook demonstrates how to load a MolFormer checkpoint file and use it to output pre-trained embeddings (without fine-tuning) given a list of SMILES strings. These can then be used with a simple model to perform a classification task.

## Load checkpoint

In [1]:
from argparse import Namespace
import yaml

with open('../../data/Pretrained_MoLFormer/hparams.yaml', 'r') as f:
    config = Namespace(**yaml.safe_load(f))
config

Namespace(accelerator='ddp', batch_size=64, beam_size=0, checkpoint_every=5000, clip_grad=50, config_load=None, config_save=None, d_dropout=0.2, data_path='', data_root='/dccstor/medscan7/smallmolecule/runs/ba-predictor/small-data/affinity', dataset_length=None, dataset_name='sol', debug=True, device='cuda', dropout=0.1, eval_every=1000, fast_dev_run=False, fc_h=512, finetune_path='', freeze_model=False, from_scratch=False, gen_save=None, gpus=8, grad_acc=1, log_file=None, lr=0.001, lr_end=0.00030000000000000003, lr_multiplier=8, lr_start=3e-05, max_epochs=4, max_len=202, measure_name='measure', min_len=1, mode='cls', model_arch='BERT_16GPU_Both_10percent_rotate_no_masking', model_load=None, model_save='model.pt', model_save_dir='./models_dump/', n_batch=1800, n_embd=768, n_head=12, n_jobs=1, n_last=1000, n_layer=12, n_samples=None, n_workers=8, nucleus_thresh=0.9, num_epoch=1, num_feats=32, num_nodes=1, num_seq_returned=0, num_workers=0, pretext_size=0, q_dropout=0.5, restart_path='',

In [2]:
from tokenizer.tokenizer import MolTranBertTokenizer

tokenizer = MolTranBertTokenizer('bert_vocab.txt')

In [3]:
from train_pubchem_light import LightningModule

ckpt = '../../data/Pretrained_MoLFormer/checkpoints/N-Step-Checkpoint_3_30000.ckpt'
lm = LightningModule(config, tokenizer.vocab).load_from_checkpoint(ckpt, config=config, vocab=tokenizer.vocab)

Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


Global seed set to 12345


Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


Global seed set to 12345


Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


## Run inference (get embeddings)

note: this runs on CPU

In [4]:
import torch
from fast_transformers.masking import LengthMask as LM

def batch_split(data, batch_size=64):
    i = 0
    while i < len(data):
        yield data[i:min(i+batch_size, len(data))]
        i += batch_size

def embed(model, smiles, tokenizer, batch_size=64):
    model.eval()
    embeddings = []
    for batch in batch_split(smiles, batch_size=batch_size):
        batch_enc = tokenizer.batch_encode_plus(batch, padding=True, add_special_tokens=True)
        idx, mask = torch.tensor(batch_enc['input_ids']), torch.tensor(batch_enc['attention_mask'])
        with torch.no_grad():
            token_embeddings = model.blocks(model.tok_emb(idx), length_mask=LM(mask.sum(-1)))
        # average pooling over tokens
        input_mask_expanded = mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embeddings.append(embedding.detach().cpu())
    return torch.cat(embeddings)

## Use linear head for classification task

There are many ways to use the embeddings for downstream tasks, this is clearly a toy example.

In [11]:
import pandas as pd

df = pd.read_csv('/home/wt/Drug_splicing/scripts/M2UMol/drug_smiles.csv')#.sample(frac=0.1)  # speed things up...
df

,smiles,id
0,O=C1N=C(N=CN1C2OC(CO)C(O)C2)N,Drug_1
1,O=C(C=1C=CC=CC1)C=2SC(=NC2N)NC(C)C,Drug_2
2,O=C(NC1=CC=CC(=C1)N2C(=O)N(C(=O)C3=C(NC4=CC=C(...,Drug_3
3,O=C1C(C(=O)C)=C(C=2C=NC(=NC2N1C3CCCC3)NC4=NC=C...,Drug_4
4,O=C(NC1=CC=C(C(=C1)NC=2N=CC=C(N2)C3=CN=CC=C3)C...,Drug_5
...,...,...
243,O=C(O)CC=1C=CC=C(OCCCN(CC2=CC=CC(=C2Cl)C(F)(F)...,Drug_245
244,O=C1SC(C(=O)N1)CC2=CC=C(OCCN(C3=NC=CC=C3)C)C=C2,Drug_246
245,O=S(=O)(C=1C=CC=CC1)N(C2=CC=C(C=C2)C(O)(C(F)(F...,Drug_247
246,O=C(N=C(N)N)C1=NC(Cl)=C(N=C1N)N,Drug_248


In [13]:
from rdkit import Chem
from sklearn.linear_model import LogisticRegression

def canonicalize(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True, isomericSmiles=False)

smiles = df.smiles.apply(canonicalize)
X = embed(lm, smiles, tokenizer).numpy()

In [14]:
dt = pd.DataFrame(X.T, columns=df.id) 
dt = dt.reset_index(drop=True)

In [15]:
dt

id,Drug_1,Drug_2,Drug_3,Drug_4,Drug_5,Drug_6,Drug_7,Drug_8,Drug_9,Drug_10,...,Drug_240,Drug_241,Drug_242,Drug_243,Drug_244,Drug_245,Drug_246,Drug_247,Drug_248,Drug_249
0,0.184446,-0.621937,-0.357980,-0.007405,0.051573,0.327295,-0.377576,0.546968,0.350073,0.649202,...,0.101864,0.194194,-0.404323,-0.412263,0.812517,0.551573,-0.153243,0.251281,-0.073279,0.517344
1,0.282411,0.267733,-0.172295,-0.170348,0.358064,0.348305,0.008289,-0.482991,-0.419295,0.647203,...,-0.152240,0.193661,0.678847,-0.028897,0.073927,0.154776,-0.099165,0.284996,0.539940,-0.311787
2,-0.228381,0.550306,0.770707,0.525418,0.141109,1.071347,0.663869,0.087786,0.718809,0.415820,...,0.361345,0.442191,0.262853,0.511448,0.162304,0.411435,0.430319,-0.228181,0.818408,0.788196
3,0.884180,0.491958,0.323343,0.277526,0.387367,0.742601,0.292110,0.097257,0.064563,-0.154933,...,0.214559,0.626686,-0.025033,0.258710,0.136983,0.247881,0.420831,0.047197,0.914285,0.437053
4,-0.916196,-0.141018,0.134376,-0.516765,-0.375931,0.469199,0.236404,-0.892173,-0.887715,-0.582865,...,-0.724575,-0.807457,-0.012748,-0.416081,-1.121679,0.044248,-0.687672,-0.198914,-0.318473,-0.437013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,-0.105630,0.445198,0.515291,-0.725298,-0.437483,0.337674,-0.034957,-0.440636,-0.665140,-0.214063,...,-0.254391,0.077928,-0.203198,-0.587014,-0.156329,0.020328,-0.176081,0.291346,-0.112486,0.088665
764,-2.569902,-2.876454,-2.372338,-3.049619,-3.582055,-2.268636,-3.266287,-2.154813,-2.439212,-3.075494,...,-2.305251,-2.205257,-2.619820,-2.436990,-3.099108,-2.873770,-3.034754,-2.001001,-2.341235,-2.090786
765,0.322760,0.167781,0.309086,-0.175663,0.132849,0.347143,0.160623,0.345388,0.034492,0.447834,...,-0.152097,-0.006007,0.370322,0.060507,0.153303,-0.110299,0.248598,0.022498,-0.244853,0.165903
766,-0.540646,-0.271466,0.052599,-0.019345,-0.233171,-0.144334,-0.039131,0.065178,-0.067102,-0.335879,...,-0.327535,-0.664056,0.141578,-0.155074,-0.215874,0.060680,-0.069494,-0.107317,-0.451943,-0.069351


In [16]:
dt.to_csv("/home/wt/Drug_splicing/data/drug_molformer_emb.csv")

In [17]:
from scipy.spatial.distance import pdist, squareform

In [18]:
###计算相似性
# 计算两两列的 cosine 距离
dist_matrix = pdist(dt.T, metric="cosine")
# 转为 DataFrame
cos_sim = 1 - squareform(dist_matrix)
cos_sim_df = pd.DataFrame(cos_sim, index=dt.columns, columns=dt.columns)

In [19]:
cos_sim_df.to_csv("/home/wt/Drug_splicing/data/drug_molformer_sim.csv")